In [109]:
prompt = "Indeed Ankara was proclaimed the capital in "

next_token ="1923"

output = "Indeed Ankara was proclaimed the capital in 1923"

In [110]:
data = "Indeed Ankara was proclaimed the capital"
hedef ="was proclaimed the capital in 1923"

In [111]:
ids = tokenizer.encode(hedef)

len(ids),ids

(12, [52, 2, 120, 105, 2, 14, 2, 48, 2, 33, 2, 127])

In [112]:
context_length = 12 #gpt-4o 128k, gemma-3-1b 32k

In [113]:
from tokenizer import Tokenizer

tokenizer = Tokenizer("tokenizer.json")
ids = tokenizer.encode(prompt)

len(ids),ids

(14, [126, 2, 77, 2, 52, 2, 120, 105, 2, 14, 2, 48, 2, 33])

In [114]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM
import os

hf_token = os.getenv("HF_TOKEN")
model_id = "google/gemma-3-1b-it"

gemma_tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
gemma_model = AutoModelForCausalLM.from_pretrained(model_id, token=hf_token)
print("ok")

Loading weights: 100%|██████████| 340/340 [00:00<00:00, 5055.08it/s]


ok


In [115]:
gemma_model

Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 1152, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=1152, out_features=1024, bias=False)
          (k_proj): Linear(in_features=1152, out_features=256, bias=False)
          (v_proj): Linear(in_features=1152, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=1152, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (up_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (down_proj): Linear(in_features=6912, out_features=1152, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma3RMSNorm((1152,), e

In [116]:
girdi = "Indeed Ankara was proclaimed the capital"
cikti = "was proclaimed the capital in 1923"

In [117]:
with open("text.txt","r") as f:
    text = f.read()

text

'This work is addressing the Capital issue as a symbol of nation-state building, which contains the founding philosophy of the Republic of Turkey after the victory in the war of independence (Milli MÃ¼cadele). Especially Mustafa Kemal did not want to continue with Istanbul as the capital of the new Republic, since it was representing the Ottoman Empire. Thus, the fundamental reasons underlying this choice was the desire to establish a new state in creation of whose identity a capital would play a defining role, as well as the troublesome situation that the Istanbul was in. Even though Ankara had already begun bearing the responsibility of a capital ever since the Representative Committee (Heyet-i Temsiliye) was moved from Sivas to Ankara, it was not expecting to be a capital, considering it a temporary situation. On the other hand, Gazi Mustafa Kemal had given the impression at the very outset that he wanted to see Ankara as the new capital. Particularly after the liberation of Istanbu

In [118]:
token_ids = tokenizer.encode(text)
#save ids
ids_text =""

for token_id in token_ids:
    ids_text += f"{token_id} "

with open("token_ids.txt","w") as f:
    f.write(ids_text)

In [119]:
import torch
from torch.utils.data import DataLoader
from text_dataset import TextDataset

stride = 5

def create_data_loader(token_ids: list, context_length: int,
                        batch_size: int, stride: int, shuffle: bool = True, device: str = "cpu"):
    dataset = TextDataset(token_ids, context_length, stride)
    data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                            generator=torch.Generator().manual_seed(42))
    return data_loader

In [120]:
train_data_loader = create_data_loader(token_ids, context_length, batch_size=4, stride=stride, shuffle=True, device="cpu")
len(train_data_loader)

37

In [121]:
for batch in enumerate(train_data_loader):
    print(batch)
    break
#batch_size kadar input ve target

(0, [tensor([[ 14,   2, 137,   2, 138,   4,   2, 139,   4,   2, 140,   2],
        [  2,  48,   2,  83,   2,  50,   2,  14,   2,  84,   2,  85],
        [ 60,   2,  61,   2,  52,   2,  14,   2,  62,   2,  44,   2],
        [148,   4,   2,  39,   2,  40,   2,  52,   2, 149,   1,   2]]), tensor([[  2, 137,   2, 138,   4,   2, 139,   4,   2, 140,   2, 115],
        [ 48,   2,  83,   2,  50,   2,  14,   2,  84,   2,  85,   2],
        [  2,  61,   2,  52,   2,  14,   2,  62,   2,  44,   2,  63],
        [  4,   2,  39,   2,  40,   2,  52,   2, 149,   1,   2,  33]])])
